# PromptOps Project Overview

This notebook explains the **PromptOps** project at a high level and also walks through the main files, components, and execution flow found in this repository.

## What this project is

PromptOps is a Python-based platform built around the idea of combining:

- prompt template management,
- dialogue summarization,
- evaluation with ROUGE metrics,
- CI/CD style quality gates,
- and a FastAPI web dashboard for interacting with all of the above.

In practice, the repository acts like a small MLOps-style system for an LLM application. It does not only run inference, but also includes the training code, evaluation code, prompt storage, testing, and deployment assets.

The project centers on **dialogue summarization**. A user submits a conversation, and the system can summarize it using either:

- a **custom Seq2Seq Transformer** trained by the team, or
- a **pretrained Hugging Face BART baseline**.

It also includes a **Q&A endpoint** that answers questions about a dialogue using a rule-based parser first, then a Flan-T5 fallback model.


## Core idea of the repository

The repository combines three layers:

1. **Application layer**
   FastAPI routes, HTML pages, static assets, API schemas, prompt loading, and history storage.

2. **ML layer**
   Model configuration, dataset loading, training logic, checkpoint loading, and inference.

3. **Operations layer**
   Evaluation, quality gates, tests, Docker packaging, Kubernetes manifests, and CI/CD hooks.

That makes PromptOps more than a single model demo. It is closer to a full project lifecycle example: **train -> evaluate -> serve -> monitor/deploy**.


In [ ]:
from pathlib import Path

root = Path('.').resolve()
print('Project root:', root)
print('\nTop-level items:')
for path in sorted(root.iterdir()):
    print('-', path.name)


## Main folders and what they contain

### `app/`
This is the web application layer.

- `main.py` creates the FastAPI app, mounts static files, includes routers, and exposes `/health`.
- `routes/api.py` contains the JSON API endpoints for summarization, QA, metrics, settings, evaluation data, CI/CD logs, and history.
- `routes/dashboard.py` maps the browser pages like `/`, `/evaluation`, `/prompts`, `/models`, `/settings`, and `/cicd`.
- `schemas.py` defines Pydantic request/response models.
- `loader.py` loads YAML prompt templates from the `prompts/` directory.
- `database.py` connects to MongoDB and exposes the history collection.
- `templates/` and `static/` support the dashboard UI.

### `model/`
This is the machine learning layer.

- `config.py` defines the central hyperparameter configuration through `ModelConfig`.
- `dataset.py` loads SAMSum and DialogSum, tokenizes them with BERT tokenizer, and builds PyTorch data loaders.
- `transformer.py` implements the Seq2Seq Transformer model.
- `train.py` contains the end-to-end training loop, validation logic, checkpoint saving, and checkpoint resume support.
- `inference.py` loads checkpoints and performs greedy or beam-search decoding.

### `scripts/`
CLI utility scripts.

- `train_model.py` is the command-line entrypoint for training.
- `quality_gate.py` evaluates the trained checkpoint and fails if the ROUGE-L score is below a threshold.

### `eval/`
Evaluation utilities and latest recorded results.

- `evaluator.py` computes ROUGE-1, ROUGE-2, and ROUGE-L.
- `results.json` stores the latest evaluation output that the dashboard can read.

### `prompts/`
YAML prompt templates with versioning-like metadata.

### `tests/`
Automated tests for API, model behavior, evaluation, and health checks.

### `k8s/`
Deployment manifests for Kubernetes.


In [ ]:
important_files = [
    'app/main.py',
    'app/routes/api.py',
    'model/config.py',
    'model/train.py',
    'model/inference.py',
    'scripts/quality_gate.py',
    'eval/results.json',
    'Dockerfile',
]

for rel in important_files:
    path = root / rel
    print(f'{rel}:', 'exists' if path.exists() else 'missing')


## How the application works

At runtime, the FastAPI application acts as the orchestrator. The high-level flow looks like this:

1. The server starts from `app/main.py`.
2. Static files and HTML templates are mounted so the dashboard can be served.
3. API routes are registered from `app/routes/api.py`.
4. A user interacts with the dashboard or directly calls the REST API.
5. The request is validated using Pydantic schemas from `app/schemas.py`.
6. Depending on the endpoint, the app either:
   - loads prompt metadata,
   - runs summarization,
   - answers a question about dialogue,
   - fetches evaluation metrics,
   - or returns CI/CD and system status information.
7. Some requests are stored in MongoDB as history records.

This architecture keeps the project modular. The API layer does not implement the entire model logic itself; instead, it calls functions from the `model/` and `eval/` packages.


## Summarization workflow

The most important feature is dialogue summarization.

When a client sends a request to `POST /api/summarize`, the app checks `model_choice`.

### Path A: Pretrained baseline
If `model_choice == 'pretrained'`, the app lazily loads the model `philschmid/bart-large-cnn-samsum` from Hugging Face.

Then it:

- tokenizes the dialogue,
- chooses generation settings based on `length_profile`,
- runs beam search generation through the pretrained model,
- decodes the result into text,
- stores the request in history,
- and returns the summary.

### Path B: Custom model
If the custom model is selected, the API expects a checkpoint at `checkpoints/best_model.pt`.

Then it:

- loads the custom Seq2Seq Transformer through `model.inference.load_model`,
- tokenizes the dialogue using `bert-base-uncased`,
- generates a summary using greedy decoding or beam search,
- decodes the output tokens,
- stores the interaction in history,
- and returns the result.

If the checkpoint does not exist, the API returns a fallback message instead of failing silently.


## Q&A workflow

The Q&A endpoint `POST /api/qa` is a nice example of a layered inference strategy.

It does **not** immediately call a language model. Instead, it first tries to answer efficiently using dialogue structure.

### Stage 1: Rule-based speaker parsing
The app parses the dialogue into speaker turns using a regular expression. This makes it possible to answer questions like:

- who started the meeting,
- who raised a concern,
- who mentioned a topic,
- or who spoke last.

### Stage 2: Keyword-based matching
If the first pass does not find a direct answer, the app scans turns for keywords extracted from the question and tries to identify the most relevant speaker or sentence.

### Stage 3: Flan-T5 fallback
Only if the rule-based logic is not enough does the application lazily load `google/flan-t5-base` and run generative QA.

This is a smart design choice because it reduces cost and latency for simpler questions while still preserving a model-based fallback for harder ones.


## Model design and training strategy

The custom model is a **Seq2Seq Transformer**. The configuration is centralized in `model/config.py` through the `ModelConfig` dataclass.

Important defaults include:

- `d_model = 512`
- `n_heads = 8`
- `n_encoder_layers = 6`
- `n_decoder_layers = 6`
- `d_ff = 2048`
- `max_seq_len = 256`
- `vocab_size = 30522`
- label smoothing enabled

The project also supports a **pretrained encoder mode**. In that setup, BERT is used as the encoder and a custom Transformer decoder is kept on top. The encoder can be frozen for a few epochs before fine-tuning begins.

That gives the project a progression across training phases:

1. Phase 1: from-scratch training on SAMSum.
2. Phase 2: training on SAMSum + DialogSum.
3. Phase 3: BERT-based encoder with custom decoder.

This phased setup is useful academically because it shows iterative model improvement rather than a single fixed experiment.


## Datasets and preprocessing

The data pipeline in `model/dataset.py` uses Hugging Face datasets.

Supported datasets:

- `knkarthick/samsum`
- `knkarthick/dialogsum`

For each training example, the system prepares:

- `encoder_input`: tokenized dialogue,
- `decoder_input`: shifted-right summary tokens for teacher forcing,
- `labels`: target summary tokens.

This is standard Seq2Seq training behavior. It means the decoder learns to predict the next token of the summary given both the dialogue and the previous summary tokens.

The project also supports combining datasets using `ConcatDataset`, which is especially relevant for Phase 2 training.


## Training loop behavior

The training logic in `model/train.py` includes several strong engineering choices:

- automatic device selection (`cuda` if available, otherwise CPU),
- checkpoint saving,
- checkpoint resume support,
- validation after each epoch,
- gradient clipping,
- label smoothing,
- warmup + cosine learning-rate scheduling,
- and differential learning rates when using the pretrained encoder.

When BERT is enabled, the encoder can stay frozen for the first few epochs and then unfreeze. This is a common transfer learning strategy that helps stabilize early training.

The best validation model is saved as `checkpoints/best_model.pt`, which becomes the central artifact later used for inference and quality gating.


## Evaluation and quality gate

PromptOps includes an evaluation layer designed to act like an MLOps checkpoint.

### ROUGE evaluation
`eval/evaluator.py` computes:

- ROUGE-1
- ROUGE-2
- ROUGE-L

These metrics are commonly used for summarization quality because they compare generated summaries with reference summaries.

### Quality gate
`scripts/quality_gate.py` loads the checkpoint, runs summarization on a test split, computes ROUGE, and checks whether ROUGE-L is above a threshold.

If the model underperforms, the script exits with a failure code. That behavior is exactly what allows the project to integrate evaluation into a CI/CD pipeline.

So instead of testing only whether the code runs, PromptOps also tests whether the model still meets a minimum quality bar.


In [ ]:
import json
from pathlib import Path

results_path = Path('eval/results.json')
if results_path.exists():
    results = json.loads(results_path.read_text(encoding='utf-8'))
    print('Latest evaluation results:')
    for key, value in results.items():
        print(f'- {key}: {value}')
else:
    print('No eval/results.json file found.')


## Current recorded evaluation snapshot

From the repository's `eval/results.json`, the latest stored metrics show:

- Phase: 3
- Model name: `BERT-Seq2Seq Phase 3`
- ROUGE-1: `0.3326`
- ROUGE-2: `0.1141`
- ROUGE-L: `0.2797`
- Validation loss: `3.8094`
- Epochs trained: `8`
- Quality gate: `PASS`
- Threshold: `0.10`

This indicates that the project already contains a saved evaluation state and that the quality gate was satisfied for the recorded run.


## Prompt management in the project

Although the project focuses heavily on summarization models, it also includes a prompt management concept.

The file `app/loader.py` reads YAML prompt files from `prompts/` and validates them using the `PromptTemplate` schema in `app/schemas.py`.

This gives prompt files a structured format with fields like:

- `name`
- `version`
- `template`
- `input_variables`
- `metadata`

That means prompts are treated more like versioned assets than loose strings inside source code, which aligns well with the "PromptOps" idea.


In [ ]:
import yaml
from pathlib import Path

prompt_path = Path('prompts/example.yaml')
if prompt_path.exists():
    prompt_data = yaml.safe_load(prompt_path.read_text(encoding='utf-8'))
    print(prompt_data)
else:
    print('Prompt file not found.')


## Dashboard and UI purpose

The dashboard layer is important because it makes the project easier to demonstrate.

Instead of exposing only API endpoints, the repository includes browser pages for:

- summarization,
- prompt management,
- evaluation results,
- model information,
- CI/CD logs,
- and settings.

This is useful for presentations, demos, and academic explanation because it turns the backend and model artifacts into a navigable interface.

The dashboard routes themselves are simple. They primarily render Jinja templates, while the actual dynamic data is fetched from the API layer.


## CI/CD and GitHub integration

The repository is designed with CI/CD in mind.

Based on the README and API routes, the project aims to support:

- linting,
- automated tests,
- model quality checks,
- and visibility into recent GitHub Actions runs.

The endpoint `/api/cicd/logs` calls the GitHub Actions API and returns recent workflow runs for the repository. This is then surfaced in the dashboard.

This is one of the most distinctive parts of the project, because it links model experimentation with software delivery practices.


## Deployment approach

The project supports both containerized and Kubernetes-style deployment.

### Docker
The `Dockerfile`:

- starts from `python:3.11-slim`,
- installs Python dependencies from `requirements.txt`,
- copies the runtime folders into `/app`,
- exposes port `8000`,
- defines a health check using `/health`,
- and launches the app through Uvicorn.

### Kubernetes
The `k8s/` folder includes manifests for namespace, deployment, service, and ingress. This suggests the project is prepared for local cluster deployment or future production-style hosting.

Together, these files show that PromptOps is not only meant to run on a laptop but also to be deployable in a more operations-oriented environment.


## What makes this project interesting

From a systems perspective, PromptOps is interesting because it merges several concerns that are often split apart:

- prompt management,
- deep learning model development,
- API serving,
- evaluation-driven validation,
- UI-based observability,
- and deployment automation.

From an academic perspective, it also shows progression across multiple model phases and compares a custom model against a strong pretrained baseline.

From a software engineering perspective, it demonstrates modular separation between app code, ML code, evaluation code, and deployment code.


## A simple mental model for the whole system

A concise way to understand PromptOps is:

- **Input**: dialogue text, prompt templates, and model checkpoints
- **Processing**: summarization, QA, evaluation, and metric collection
- **Control layer**: FastAPI routes, schemas, prompt loader, quality gate logic
- **Persistence**: checkpoints, evaluation results, prompt YAML files, MongoDB history
- **Output**: summaries, answers, evaluation metrics, dashboard pages, and CI/CD status

So the project is best seen as an end-to-end **LLM application operations pipeline**, not just a standalone model.


## Final summary

PromptOps is a full-stack ML application for dialogue summarization and prompt-oriented operations. It includes:

- a FastAPI backend,
- a dashboard UI,
- a custom Transformer training pipeline,
- pretrained baseline inference,
- a QA subsystem,
- ROUGE-based evaluation,
- CI/CD-style quality gates,
- and Docker/Kubernetes deployment support.

If you need to explain this project in one sentence, a strong summary would be:

**PromptOps is an MLOps-style platform for managing, evaluating, and serving dialogue summarization workflows with prompt templates, custom models, quality gates, and deployment support.**
